# Text mining: Exercises 6, 7, 8 and 9

🤔 **Can a computer tell which movies are similar, just by reading their description?**

In exercise 2, you saved one document per movie (title, synopsis and actors). In this notebook, we compare four of them:
- Harry Potter 1,
- Harry Potter 2,
- The Lord of the Rings 1,
- The Hobbit 1.

A computer cannot compare texts directly: it compares **numbers**. So we will turn each document into a **vector**, then measure how close the vectors are.

| Exercise | What we do | Tool |
|---|---|---|
| 6 | Clean the texts and cut them into **tokens** | `nltk` |
| 7 | Turn each document into a vector with **TF-IDF** | Python + `numpy` |
| 8 | Turn each document into an **embedding** with Doc2Vec and SBERT | `gensim`, `sentence-transformers` |
| 9 | Compare the vectors with the **cosine similarity**, and search the movies by meaning | `numpy` |

⚠️ **Before starting**: run the "Going further" part of **exercise 2** (`1_data_collection/exercise2.ipynb`). It creates the files `doc_1.txt` to `doc_4.txt` used here.

### How to work

- Run the cells **one by one, from top to bottom** (`Shift + Enter`).
- ✏️ marks a cell you have to complete: replace the lines marked `## @COMPLETE`.
- 🧪 marks a test cell: run it to check your answer before moving on.
- 💡 gives you a hint, 🤔 asks you a question to think about.

In [ ]:
import math
from collections import Counter
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer

# Download the resources used by nltk (only the first time, then it is instantaneous)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

DOCS_PATH = Path("../1_data_collection/docs")

## Load the documents

Each file `doc_<i>.txt` contains the title of a movie on its first line, then the text of the document. We store the corpus in a dictionary: `title -> text`.

In [ ]:
def load_corpus(path):
    """Read the files doc_1.txt, doc_2.txt, ... and return a dictionary title -> text."""
    corpus = {}
    for file_path in sorted(path.glob("doc_[0-9]*.txt")):
        title, text = file_path.read_text(encoding="utf-8").split("\n", 1)
        corpus[title] = text
    return corpus


corpus = load_corpus(DOCS_PATH)

for title, text in corpus.items():
    print("=====", title)
    print(text[:200], "...")

## Exercise 6: Token extraction

Before turning a text into numbers, we clean it so that only the **meaningful words** remain, in a standard form. We will apply four steps to each text. Let's try them first on a single sentence:

In [ ]:
sentence = "Harry Potter is learning magic with his friends at Hogwarts, the school of wizards!"

### Step 1: Lowercase and tokenize

**Tokenizing** means cutting a text into a list of words and punctuation signs, called **tokens**. We also put everything in lowercase so that `Harry` and `harry` are the same token.

<img src="img/tokenization.png" width="560" alt="Tokenization: from raw text to tokens">

```python
"Hello World".lower()                    # 'hello world'
nltk.word_tokenize("hello world!")       # ['hello', 'world', '!']
```

✏️ **Complete the function `tokenize`.**

In [ ]:
def tokenize(text):
    """Return the list of tokens of a text, in lowercase."""
    ## @COMPLETE : put the text in lowercase, then cut it into tokens
    return []

🧪 **Test**: you should see `['harry', 'potter', 'is', 'learning', ...]`, with the comma and the `!` as separate tokens.

In [ ]:
print(tokenize(sentence))

### Step 2: Remove punctuation and stopwords

Some tokens carry no meaning:
- **punctuation**: `,`, `!`, `'s`, ...
- **stopwords**: very common words such as `the`, `is`, `with`, `his`, ... They appear in every text, so they do not help to tell texts apart.

`nltk` provides a list of English stopwords (run the cell below to see some of them). To detect punctuation, the string method `.isalpha()` returns `True` only if a string is made of letters.

In [ ]:
STOPWORDS = set(stopwords.words("english"))

print(len(STOPWORDS), "stopwords, for instance:", sorted(STOPWORDS)[:15])

✏️ **Complete the function `is_useful`**: a token is useful if it is made of letters **and** is not a stopword.

In [ ]:
def is_useful(token):
    """Return True if the token is a word (only letters) that is not a stopword."""
    ## @COMPLETE : return True or False
    return True

🧪 **Test**: you should only see meaningful words: `['harry', 'potter', 'learning', 'magic', 'friends', 'hogwarts', 'school', 'wizards']`.

In [ ]:
print([token for token in tokenize(sentence) if is_useful(token)])

### Step 3: Stemming

`wizard` and `wizards`, `learn` and `learning` have the same meaning but are different tokens. **Stemming** cuts the end of the words to keep only their **stem** (the result is not always a real word, and that's fine):

In [ ]:
stemmer = PorterStemmer()

for word in ["wizard", "wizards", "learning", "learned", "friends", "magical"]:
    print(word, "->", stemmer.stem(word))

### Step 4: Put it all together

✏️ **Complete the function `preprocess`**: tokenize the text, keep only the useful tokens, and return their stems.

💡 *Hint*: a list comprehension with a condition does it in one line: `[... for token in ... if ...]`.

In [ ]:
def preprocess(text):
    """Return the list of stems of the useful tokens of a text."""
    ## @COMPLETE : tokenize, keep the useful tokens, and stem them
    return []

🧪 **Test**: you should see `['harri', 'potter', 'learn', 'magic', 'friend', 'hogwart', 'school', 'wizard']`.

In [ ]:
print(preprocess(sentence))

Now, let's apply it to the whole corpus:

In [ ]:
tokens = {title: preprocess(text) for title, text in corpus.items()}

for title, doc_tokens in tokens.items():
    print(title, ":", len(doc_tokens), "tokens, for instance", doc_tokens[:8])

### Which tokens appear in many documents?

The **document frequency** of a token is the number of documents in which it appears. Run the cell below to see the most frequent tokens.

In [ ]:
document_frequency = Counter()
for doc_tokens in tokens.values():
    document_frequency.update(set(doc_tokens))  # set(): count each token once per document

print("Number of different tokens:", len(document_frequency))
print("Most frequent tokens:", document_frequency.most_common(10))

🤔 **Question**: a token that appears in the four documents, does it help to tell the movies apart? And a token that appears in only one document?

## Exercise 7: Vectorization with TF-IDF

The idea of TF-IDF is to represent each document by a vector with **one value per token of the vocabulary**. The value of a token is high if the token is **frequent in this document** but **rare in the other documents**: it is then characteristic of the document.

It is the product of two factors:

- **TF** (*term frequency*): how frequent the token is in the document

$$TF(t, d) = \frac{\text{number of times } t \text{ appears in } d}{\text{number of tokens in } d}$$

- **IDF** (*inverse document frequency*): how rare the token is in the corpus

$$IDF(t) = \log\left(\frac{\text{number of documents}}{\text{number of documents containing } t}\right)$$

💡 A token present in **all** the documents gets $IDF = \log(1) = 0$: it is ignored.

✏️ **Complete the functions `tf` and `idf`.**

💡 *Hint*: `doc_tokens.count(token)` counts how many times `token` appears in the list `doc_tokens`, and the dictionary `document_frequency` (computed above) gives the number of documents containing a token.

In [ ]:
def tf(token, doc_tokens):
    """Term frequency of a token in a document (given as its list of tokens)."""
    ## @COMPLETE : return the relative frequency of the token in the document
    return 0


def idf(token):
    """Inverse document frequency of a token in the corpus."""
    nb_docs = len(tokens)
    ## @COMPLETE : return the IDF of the token (use math.log)
    return 0

🧪 **Test**: `potter` appears in two documents out of four, so its IDF is $\log(4/2) \approx 0.69$.

In [ ]:
print("IDF of 'potter':", round(idf("potter"), 2))
print("TF of 'potter' in Harry Potter 1:", round(tf("potter", tokens["Harry Potter 1"]), 3))

Now we can build the vectors. The **vocabulary** is the list of all the different tokens of the corpus: each token is one dimension of the vectors. We store the vectors in a table (a DataFrame) with **one row per movie** and **one column per token**.

In [ ]:
vocabulary = sorted(document_frequency)

tfidf = pd.DataFrame(
    [[tf(token, doc_tokens) * idf(token) for token in vocabulary] for doc_tokens in tokens.values()],
    index=tokens.keys(),
    columns=vocabulary,
)

print("Vectors of", tfidf.shape[1], "dimensions")
tfidf.round(3)

Most values are `0`: each document only uses a small part of the vocabulary. Let's look at the **five most characteristic tokens** of each movie (the highest TF-IDF values):

In [ ]:
for title, vector in tfidf.iterrows():
    print(title, ":", list(vector.sort_values(ascending=False).index[:5]))

## Exercise 8: Embeddings with Doc2Vec and SBERT

TF-IDF vectors are **very long** (one dimension per token) and only count words: `wizard` and `sorcerer` are two unrelated dimensions, even if they mean almost the same thing.

An **embedding** is a **short, dense vector** learned by a model, in which texts (or words) with a **close meaning** end up **close in space**:

<img src="img/embeddings.png" width="560" alt="Embeddings: close in meaning, close in space">

We will build embeddings in two ways:
- **8a, Doc2Vec**: we train a small model **ourselves**, on our four documents,
- **8b, SBERT**: we use a large model **already trained** on millions of sentences.

### 8a: Doc2Vec

**Doc2Vec** is a small neural network that learns a **short vector** (here 5 dimensions) for each document, by trying to predict the words of the document. The dimensions no longer correspond to tokens: they are learned by the model.

With `gensim`:
1. each document is given as a `TaggedDocument(words=list_of_tokens, tags=[identifier])`,
2. the model is trained with `Doc2Vec(documents, vector_size=..., min_count=..., epochs=...)`,
3. the vector of document `i` is `doc2vec_model.dv[i]`.

✏️ **Create and train a Doc2Vec model called `doc2vec_model`, with vectors of 5 dimensions**, keeping all the tokens (`min_count=1`).

💡 *Hint*: also pass `epochs=100` (number of training passes), and `seed=42, workers=1` so that the results are the same every time you run the cell.

In [ ]:
documents = [TaggedDocument(words=doc_tokens, tags=[i]) for i, doc_tokens in enumerate(tokens.values())]

## @COMPLETE : create and train the model
doc2vec_model = None

🧪 **Test**: you should see a table with one row per movie and 5 columns.

In [ ]:
doc2vec = pd.DataFrame([doc2vec_model.dv[i] for i in range(len(documents))], index=tokens.keys())

doc2vec.round(3)

⚠️ Doc2Vec needs **a lot of text** to learn meaningful vectors. With only four short documents, its results are fragile: keep this in mind in exercise 9.

### 8b: SBERT (Sentence-BERT)

**BERT** is a large language model that reads a text **in context**: the meaning of a word depends on its neighbours. **SBERT** is a version of BERT trained so that two sentences with a close meaning get close vectors, exactly what we need.

We don't train anything: we download a model **already trained** on millions of sentences (`all-MiniLM-L6-v2`, about 90 MB, downloaded the first time), and ask it to **encode** our documents.

💡 Note the difference with TF-IDF and Doc2Vec: SBERT reads the **raw text**, not our preprocessed tokens. It needs the stopwords, the word order and the punctuation to understand the sentences.

```python
sbert_model.encode(list_of_texts)   # returns one vector per text
```

✏️ **Encode the texts of the corpus with SBERT.**

💡 *Hint*: `corpus.values()` gives the texts, and `list(...)` turns them into a list.

In [ ]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

## @COMPLETE : encode the texts of the corpus
sbert_vectors = None

🧪 **Test**: you should see a table with one row per movie and 384 columns: SBERT vectors are longer than our Doc2Vec ones, but much shorter than the TF-IDF ones when the corpus is large.

In [ ]:
sbert = pd.DataFrame(sbert_vectors, index=corpus.keys())

sbert.round(3)

## Exercise 9: Corpus analysis with the cosine similarity

Two documents are similar if their vectors **point in the same direction**. The **cosine similarity** measures the angle between two vectors $u$ and $v$:

$$\cos(u, v) = \frac{u \cdot v}{\|u\| \times \|v\|}$$

- $1$: same direction (very similar documents),
- $0$: perpendicular vectors (nothing in common),
- $-1$: opposite directions (possible with embeddings, not with TF-IDF whose values are all positive).

With `numpy`: `np.dot(u, v)` is the dot product $u \cdot v$ and `np.linalg.norm(u)` is the norm $\|u\|$.

✏️ **Complete the function `cosine_similarity`.**

In [ ]:
def cosine_similarity(u, v):
    """Cosine similarity between two vectors."""
    ## @COMPLETE : return the cosine similarity between u and v
    return 0

🧪 **Test**: you should get `1.0`, `0.0` and `-1.0`.

In [ ]:
print(round(cosine_similarity(np.array([1, 2]), np.array([2, 4])), 2))   # same direction
print(round(cosine_similarity(np.array([1, 0]), np.array([0, 3])), 2))   # perpendicular
print(round(cosine_similarity(np.array([1, 1]), np.array([-1, -1])), 2)) # opposite directions

The function below computes the similarity between **every pair of documents** and returns it as a table: the cell (row A, column B) is the similarity between movie A and movie B.

In [ ]:
def similarity_matrix(vectors):
    """Return the cosine similarity between every pair of rows of a DataFrame."""
    titles = vectors.index
    return pd.DataFrame(
        [[cosine_similarity(vectors.loc[a], vectors.loc[b]) for b in titles] for a in titles],
        index=titles,
        columns=titles,
    )

### Similarities with TF-IDF

In [ ]:
similarity_matrix(tfidf).round(2)

### Similarities with Doc2Vec

In [ ]:
similarity_matrix(doc2vec).round(2)

### Similarities with SBERT

In [ ]:
similarity_matrix(sbert).round(2)

### Search the movies by meaning

Comparing movies to each other is one use. Another one is to compare them to a **query** that we write ourselves, like in a search engine. We turn the query into a vector **with the same method** as the documents, then compute its similarity with each movie.

The difficulty: the query may use **other words** than the documents. The synopsis of Harry Potter 1 speaks about a *wizard* and a *School of Witchcraft and Wizardry*, never about a *sorcerer* or an *academy*. Let's see how each method reacts.

In [ ]:
def search(query):
    """Return the cosine similarity between a query and each movie, for the three methods."""
    query_tokens = preprocess(query)

    # TF-IDF: the query is described with the vocabulary of the corpus (unknown words are ignored)
    query_tfidf = np.array([tf(token, query_tokens) * idf(token) for token in vocabulary])
    # Doc2Vec: the trained model infers a vector for a new text
    query_doc2vec = doc2vec_model.infer_vector(query_tokens)
    # SBERT: the model encodes the raw query
    query_sbert = sbert_model.encode(query)

    scores = {}
    for method, vectors, query_vector in [
        ("TF-IDF", tfidf, query_tfidf),
        ("Doc2Vec", doc2vec, query_doc2vec),
        ("SBERT", sbert, query_sbert),
    ]:
        if not query_vector.any():  # no word in common with the corpus: the similarity is 0
            scores[method] = [0.0] * len(vectors)
        else:
            scores[method] = [cosine_similarity(query_vector, vector) for vector in vectors.values]
    return pd.DataFrame(scores, index=corpus.keys()).round(2)

In [ ]:
search("a young sorcerer studying at an academy of magic")

🎬 **Your turn**: write your own query in the cell below (for instance `"a long journey to destroy a dangerous jewel"`). Which method finds the right movie?

In [ ]:
search("a long journey to destroy a dangerous jewel")

🤔 **Questions**:
- With TF-IDF, which movies are the closest to each other? Is it what you expected?
- Look again at the most characteristic tokens of exercise 7: which tokens make Harry Potter 1 and Harry Potter 2 similar? Are they related to the story or to the actors?
- Which method works best here, TF-IDF, Doc2Vec or SBERT? Why? (💡 *Hint*: think about the size of our corpus, and about the data each model has learned from.)
- Run the Doc2Vec cells again with `seed=1` instead of `seed=42`. Do the similarities change? What does it tell you?
- In the search, why does TF-IDF fail when the query uses synonyms? Why does SBERT succeed?

## 🎉 Well done!

You have built a complete text mining pipeline:
1. **preprocessing**: lowercase, tokenization, removal of punctuation and stopwords, stemming,
2. **vectorization**:

| Method | Starts from | Vector | Captures | Main limit |
|---|---|---|---|---|
| TF-IDF | the tokens | long, one dimension per token | shared words | blind to synonyms |
| Doc2Vec | the tokens | short, learned on our corpus | some meaning | needs a lot of text |
| SBERT | the raw text | short, pre-trained model | the meaning of whole sentences | large model, harder to interpret |

3. **comparison**: the cosine similarity between vectors, between documents or between a query and documents.

In **exercise 10**, you will use these tools on a real corpus of 4,000 movie reviews.